# QAOA-GPT inference demo

In [1]:
from model_interface import QAOA_GPT

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import random
from tqdm import tqdm
from collections import defaultdict, Counter
import json
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## Loading a model

In [3]:
qaoa_gpt_n10_obj = QAOA_GPT(
    model_ckpt='nanoGPT/models/n10w_qaoa_mixer/ckpt_16000_gemb__ar_0_96584__er_0_0.pt',
    data_dir='nanoGPT/models/n10w_qaoa_mixer/data', # to take meta.pkl file 
    config_file='nanoGPT/models/n10w_qaoa_mixer/data/train_adapt_gpt_config.py',
    temp_folder='temp_data',
    device='cuda'
)

Loading config from: nanoGPT/models/n10w_qaoa_mixer/data/train_adapt_gpt_config.py
Initiating nanoGPT model with padding support
number of parameters: 11.60M


## Generating random graphs

In [4]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [5]:
tqdm.pandas()

Modify this nodes here to match the model before run

In [6]:
n_graphs = 50
n_nodes = 10

In [7]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [8]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 1, {'weight': 0.09}), (0, 2, {'weight': 0.26}), (0, 3, {'weight': 0.13}), (0, 4, {'weight': 0.29}), (0, 5, {'weight': 0.71}), (0, 8, {'weight': 0.51}), (0, 9, {'weight': 0.15}), (1, 2, {'weight': 0.28}), (1, 3, {'weight': 0.16}), (1, 4, {'weight': 0.31}), (1, 5, {'weight': 0.17}), (1, 6, {'weight': 0.55}), (1, 7, {'weight': 0.17}), (1, 8, {'weight': 0.89}), (1, 9, {'weight': 0.28}), (2, 3, {'weight': 0.72}), (2, 4, {'weight': 0.39}), (2, 5, {'weight': 0.28}), (2, 7, {'weight': 0.65}), (2, 8, {'weight': 0.56}), (2, 9, {'weight': 0.55}), (3, 4, {'weight': 0.46}), (3, 6, {'weight': 0.92}), (3, 7, {'weight': 0.39}), (3, 8, {'weight': 0.4}), (3, 9, {'weight': 0.07}), (4, 5, {'weight': 0.84}), (4, 6, {'weight': 0.48}), (4, 7, {'weight': 0.59}), (4, 8, {'weight': 0.64}), (4, 9, {'weight': 0.12}), (5, 6, {'weight': 0.24}), (5, 7, {'weight': 0.22}), (5, 8, {'weight': 0.07}), (6, 7, {'weight': 0.12}), (6, 9, {'weight': 0.77}), (7, 8, {'weight': 0.13}), (8, 9, {'weight': 0.26})]

## Generate circuits with QAOA-GPT model

In [9]:
qaoa_gpt_circ_df = qaoa_gpt_n10_obj.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    # calculate_classic_maxcut=True, # to create col enery_gurobi. Default:True
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
)

Preparing graphs...:   0%|          | 0/50 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2027-11-29


Preparing graphs...: 100%|██████████| 50/50 [00:00<00:00, 155.50it/s]


Performing FEATHER embedding


100%|██████████| 50/50 [00:00<00:00, 1778.06it/s]
Inference. Current batch: n_edges: 41, n_graphs: 1: 100%|██████████| 16/16 [00:12<00:00,  1.23it/s]


In [10]:
qaoa_gpt_n10_obj.graphs_nx_df[:1]

,graph_id,elist,n_nodes,energy_gurobi,token_seq_round_d2,edgelist_list_len,approx_ratio,label,edgelist_json,has_emb
0,er_graph_0,"[(1, 2, 0.42), (1, 3, 0.5), (1, 4, 0.76), (1, 7, 0.07), (1, 10, 0.59), (2, 4, 0.33), (2, 9, 0.22), (2, 10, 0.59), (3, 6, 0.26), (3, 7, 0.46), (3, 8, 0.51), (3, 9, 0.37), (4, 7, 0.9), (4, 8, 0.66), (4, 10, 0.74), (5, 6, 0.67), (5, 8, 0.39), (6, 7, 0.97), (6, 9, 0.88), (6, 10, 0.51), (7, 8, 0.36), (7, 10, 0.94), (8, 9, 0.31), (9, 10, 0.48)]",10,-9.61,"[bos, (1, 2), 0.42, (1, 3), 0.5, (1, 4), 0.76, (1, 7), 0.07, (1, 10), 0.59, (2, 4), 0.33, (2, 9), 0.22, (2, 10), 0.59, (3, 6), 0.26, (3, 7), 0.46, (3, 8), 0.51, (3, 9), 0.37, (4, 7), 0.9, (4, 8), 0.66, (4, 10), 0.74, (5, 6), 0.67, (5, 8), 0.39, (6, 7), 0.97, (6, 9), 0.88, (6, 10), 0.51, (7, 8), 0.36, (7, 10), 0.94, (8, 9), 0.31, (9, 10), 0.48, end_of_graph]",24,None,test_interactive,"[[1, 2, 0.42], [1, 3, 0.5], [1, 4, 0.76], [1, 7, 0.07], [1, 10, 0.59], [2, 4, 0.33], [2, 9, 0.22], [2, 10, 0.59], [3, 6, 0.26], [3, 7, 0.46], [3, 8, 0.51], [3, 9, 0.37], [4, 7, 0.9], [4, 8, 0.66], [4, 10, 0.74], [5, 6, 0.67], [5, 8, 0.39], [6, 7, 0.97], [6, 9, 0.88], [6, 10, 0.51], [7, 8, 0.36], [7, 10, 0.94], [8, 9, 0.31], [9, 10, 0.48]]",True


In [11]:
qaoa_gpt_n10_obj.feather_par_emb[0][:10]

array([ 0.52,  0.51,  0.47,  0.41,  0.33,  0.24,  0.14,  0.03, -0.08,
       -0.19])

In [12]:
# qaoa_gpt_n10_obj.meta

In [13]:
# qaoa_gpt_n10_obj.emb_graph_id_to_idx_dict

In [14]:
sample_gr = graphs_edgelist_list_dict['er_graph_0'].edges(data=True)
sample_gr

EdgeDataView([(0, 1, {'weight': 0.42}), (0, 2, {'weight': 0.5}), (0, 3, {'weight': 0.76}), (0, 6, {'weight': 0.07}), (0, 9, {'weight': 0.59}), (1, 3, {'weight': 0.33}), (1, 8, {'weight': 0.22}), (1, 9, {'weight': 0.59}), (2, 5, {'weight': 0.26}), (2, 6, {'weight': 0.46}), (2, 7, {'weight': 0.51}), (2, 8, {'weight': 0.37}), (3, 6, {'weight': 0.9}), (3, 7, {'weight': 0.66}), (3, 9, {'weight': 0.74}), (4, 5, {'weight': 0.67}), (4, 7, {'weight': 0.39}), (5, 6, {'weight': 0.97}), (5, 8, {'weight': 0.88}), (5, 9, {'weight': 0.51}), (6, 7, {'weight': 0.36}), (6, 9, {'weight': 0.94}), (7, 8, {'weight': 0.31}), (8, 9, {'weight': 0.48})])

In [15]:
len(graphs_edgelist_list_dict['er_graph_0'].edges(data=True))

24

The graph after prediction is shifted by 1 unit. For example, in NetworkX the edge is (0, 2, 0.36), but in this DataFrame it becomes (1, 3, 0.36)

In [16]:
qaoa_gpt_circ_df[:1]

,graph,n_edges,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_gurobi,label,graph_w_jl,graph_weight_norm
0,"[(1, 2), 0.09, (1, 3), 0.26, (1, 4), 0.13, (1, 5), 0.29, (1, 6), 0.71, (1, 9), 0.51, (1, 10), 0.15, (2, 3), 0.28, (2, 4), 0.16, (2, 5), 0.31, (2, 6), 0.17, (2, 7), 0.55, (2, 8), 0.17, (2, 9), 0.89, (2, 10), 0.28, (3, 4), 0.72, (3, 5), 0.39, (3, 6), 0.28, (3, 8), 0.65, (3, 9), 0.56, (3, 10), 0.55, (4, 5), 0.46, (4, 7), 0.92, (4, 8), 0.39, (4, 9), 0.4, (4, 10), 0.07, (5, 6), 0.84, (5, 7), 0.48, (5, 8), 0.59, (5, 9), 0.64, (5, 10), 0.12, (6, 7), 0.24, (6, 8), 0.22, (6, 9), 0.07, (7, 8), 0.12, (7, 10), 0.77, (8, 9), 0.13, (9, 10), 0.26]",38,"[[new_layer_p, 1, -0.49, 0.35, new_layer_p, 1, -0.36, 0.71, new_layer_p, 1, -0.31, 0.76, new_layer_p, 1, -0.29, 0.86, new_layer_p, 1, -0.27, 0.92, new_layer_p, 1, -0.25, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.2, new_layer_p, 1, -0.1, 1.4], [new_layer_p, 1, -0.48, 0.35, new_layer_p, 1, -0.35, 0.72, new_layer_p, 1, -0.3, 0.77, new_layer_p, 1, -0.28, 0.87, new_layer_p, 1, -0.26, 0.94, new_layer_p, 1, -0.24, 1.02, new_layer_p, 1, -0.2, 1.13, new_layer_p, 1, -0.12, 1.32], [new_layer_p, 1, -0.48, 0.35, new_layer_p, 1, -0.35, 0.71, new_layer_p, 1, -0.3, 0.77, new_layer_p, 1, -0.28, 0.89, new_layer_p, 1, -0.26, 0.96, new_layer_p, 1, -0.23, 1.05, new_layer_p, 1, -0.18, 1.15, new_layer_p, 1, -0.1, 1.35], [new_layer_p, 1, -0.48, 0.35, new_layer_p, 1, -0.35, 0.71, new_layer_p, 1, -0.3, 0.77, new_layer_p, 1, -0.28, 0.87, new_layer_p, 1, -0.26, 0.92, new_layer_p, 1, -0.24, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.2, new_layer_p, 1, -0.1, 1.43], [new_layer_p, 1, -0.48, 0.35, new_layer_p, 1, -0.35, 0.71, new_layer_p, 1, -0.31, 0.77, new_layer_p, 1, -0.29, 0.87, new_layer_p, 1, -0.27, 0.92, new_layer_p, 1, -0.25, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.17, new_layer_p, 1, -0.1, 1.39]]",[],None,er_graph_2,-10.63,test_interactive,"[[1, 2, 0.09], [1, 3, 0.26], [1, 4, 0.13], [1, 5, 0.29], [1, 6, 0.71], [1, 9, 0.51], [1, 10, 0.15], [2, 3, 0.28], [2, 4, 0.16], [2, 5, 0.31], [2, 6, 0.17], [2, 7, 0.55], [2, 8, 0.17], [2, 9, 0.89], [2, 10, 0.28], [3, 4, 0.72], [3, 5, 0.39], [3, 6, 0.28], [3, 8, 0.65], [3, 9, 0.56], [3, 10, 0.55], [4, 5, 0.46], [4, 7, 0.92], [4, 8, 0.39], [4, 9, 0.4], [4, 10, 0.07], [5, 6, 0.84], [5, 7, 0.48], [5, 8, 0.59], [5, 9, 0.64], [5, 10, 0.12], [6, 7, 0.24], [6, 8, 0.22], [6, 9, 0.07], [7, 8, 0.12], [7, 10, 0.77], [8, 9, 0.13], [9, 10, 0.26]]",1.0


In [17]:
qaoa_gpt_circ_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   graph              50 non-null     object 
 1   n_edges            50 non-null     int64  
 2   q_circuits         50 non-null     object 
 3   adapt_circuit      50 non-null     object 
 4   adapt_full_ar      0 non-null      object 
 5   graph_prefix       50 non-null     object 
 6   energy_gurobi      50 non-null     float64
 7   label              50 non-null     object 
 8   graph_w_jl         50 non-null     object 
 9   graph_weight_norm  50 non-null     float64
dtypes: float64(2), int64(1), object(7)
memory usage: 4.0+ KB


## Evaluate circuits

In [18]:
qaoa_gpt_circ_eval_df = qaoa_gpt_n10_obj.eval_circ_df_jl(qaoa_gpt_circ_df)

In [19]:
sample_gr

EdgeDataView([(0, 1, {'weight': 0.42}), (0, 2, {'weight': 0.5}), (0, 3, {'weight': 0.76}), (0, 6, {'weight': 0.07}), (0, 9, {'weight': 0.59}), (1, 3, {'weight': 0.33}), (1, 8, {'weight': 0.22}), (1, 9, {'weight': 0.59}), (2, 5, {'weight': 0.26}), (2, 6, {'weight': 0.46}), (2, 7, {'weight': 0.51}), (2, 8, {'weight': 0.37}), (3, 6, {'weight': 0.9}), (3, 7, {'weight': 0.66}), (3, 9, {'weight': 0.74}), (4, 5, {'weight': 0.67}), (4, 7, {'weight': 0.39}), (5, 6, {'weight': 0.97}), (5, 8, {'weight': 0.88}), (5, 9, {'weight': 0.51}), (6, 7, {'weight': 0.36}), (6, 9, {'weight': 0.94}), (7, 8, {'weight': 0.31}), (8, 9, {'weight': 0.48})])

The eval df add 1 more col is adapt_gpt_energies

Each graph is generate into 5 other graphs, that why we see list of 5 q_circuits and adapt_gpt_energies



In [20]:
qaoa_gpt_circ_eval_df[:1]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_2,"[[1, 2], 0.09, [1, 3], 0.26, [1, 4], 0.13, [1, 5], 0.29, [1, 6], 0.71, [1, 9], 0.51, [1, 10], 0.15, [2, 3], 0.28, [2, 4], 0.16, [2, 5], 0.31, [2, 6], 0.17, [2, 7], 0.55, [2, 8], 0.17, [2, 9], 0.89, [2, 10], 0.28, [3, 4], 0.72, [3, 5], 0.39, [3, 6], 0.28, [3, 8], 0.65, [3, 9], 0.56, [3, 10], 0.55, [4, 5], 0.46, [4, 7], 0.92, [4, 8], 0.39, [4, 9], 0.4, [4, 10], 0.07, [5, 6], 0.84, [5, 7], 0.48, [5, 8], 0.59, [5, 9], 0.64, [5, 10], 0.12, [6, 7], 0.24, [6, 8], 0.22, [6, 9], 0.07, [7, 8], 0.12, [7, 10], 0.77, [8, 9], 0.13, [9, 10], 0.26]",38,"[[new_layer_p, 1, -0.49, 0.35000000000000003, new_layer_p, 1, -0.36, 0.71, new_layer_p, 1, -0.31, 0.76, new_layer_p, 1, -0.29, 0.86, new_layer_p, 1, -0.27, 0.92, new_layer_p, 1, -0.25, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.2, new_layer_p, 1, -0.1, 1.4], [new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.72, new_layer_p, 1, -0.30000000000000004, 0.77, new_layer_p, 1, -0.28, 0.87, new_layer_p, 1, -0.26, 0.9400000000000001, new_layer_p, 1, -0.24, 1.02, new_layer_p, 1, -0.2, 1.13, new_layer_p, 1, -0.12, 1.32], [new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.71, new_layer_p, 1, -0.30000000000000004, 0.77, new_layer_p, 1, -0.28, 0.89, new_layer_p, 1, -0.26, 0.96, new_layer_p, 1, -0.23, 1.05, new_layer_p, 1, -0.18, 1.15, new_layer_p, 1, -0.1, 1.35], [new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.71, new_layer_p, 1, -0.30000000000000004, 0.77, new_layer_p, 1, -0.28, 0.87, new_layer_p, 1, -0.26, 0.92, new_layer_p, 1, -0.24, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.2, new_layer_p, 1, -0.1, 1.43], [new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.71, new_layer_p, 1, -0.31, 0.77, new_layer_p, 1, -0.29, 0.87, new_layer_p, 1, -0.27, 0.92, new_layer_p, 1, -0.25, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.17, new_layer_p, 1, -0.1, 1.3900000000000001]]","[-10.324628931612036, -10.296200368111819, -10.329600542170912, -10.321310238690877, -10.308112351758158]",-10.63


In [21]:
qaoa_gpt_circ_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   graph_prefix        50 non-null     object 
 1   graph               50 non-null     object 
 2   n_edges             50 non-null     int64  
 3   q_circuits          50 non-null     object 
 4   adapt_gpt_energies  50 non-null     object 
 5   energy_gurobi       50 non-null     float64
dtypes: float64(1), int64(1), object(4)
memory usage: 2.5+ KB


In [22]:
# 3 extract first rows into 5 rows for 5 q_circuits and adapt_gpt_energies
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits']) 

In [23]:
qaoa_gpt_circ_eval_expl_df[:5]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_2,"[[1, 2], 0.09, [1, 3], 0.26, [1, 4], 0.13, [1, 5], 0.29, [1, 6], 0.71, [1, 9], 0.51, [1, 10], 0.15, [2, 3], 0.28, [2, 4], 0.16, [2, 5], 0.31, [2, 6], 0.17, [2, 7], 0.55, [2, 8], 0.17, [2, 9], 0.89, [2, 10], 0.28, [3, 4], 0.72, [3, 5], 0.39, [3, 6], 0.28, [3, 8], 0.65, [3, 9], 0.56, [3, 10], 0.55, [4, 5], 0.46, [4, 7], 0.92, [4, 8], 0.39, [4, 9], 0.4, [4, 10], 0.07, [5, 6], 0.84, [5, 7], 0.48, [5, 8], 0.59, [5, 9], 0.64, [5, 10], 0.12, [6, 7], 0.24, [6, 8], 0.22, [6, 9], 0.07, [7, 8], 0.12, [7, 10], 0.77, [8, 9], 0.13, [9, 10], 0.26]",38,"[new_layer_p, 1, -0.49, 0.35000000000000003, new_layer_p, 1, -0.36, 0.71, new_layer_p, 1, -0.31, 0.76, new_layer_p, 1, -0.29, 0.86, new_layer_p, 1, -0.27, 0.92, new_layer_p, 1, -0.25, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.2, new_layer_p, 1, -0.1, 1.4]",-10.324629,-10.63
0,er_graph_2,"[[1, 2], 0.09, [1, 3], 0.26, [1, 4], 0.13, [1, 5], 0.29, [1, 6], 0.71, [1, 9], 0.51, [1, 10], 0.15, [2, 3], 0.28, [2, 4], 0.16, [2, 5], 0.31, [2, 6], 0.17, [2, 7], 0.55, [2, 8], 0.17, [2, 9], 0.89, [2, 10], 0.28, [3, 4], 0.72, [3, 5], 0.39, [3, 6], 0.28, [3, 8], 0.65, [3, 9], 0.56, [3, 10], 0.55, [4, 5], 0.46, [4, 7], 0.92, [4, 8], 0.39, [4, 9], 0.4, [4, 10], 0.07, [5, 6], 0.84, [5, 7], 0.48, [5, 8], 0.59, [5, 9], 0.64, [5, 10], 0.12, [6, 7], 0.24, [6, 8], 0.22, [6, 9], 0.07, [7, 8], 0.12, [7, 10], 0.77, [8, 9], 0.13, [9, 10], 0.26]",38,"[new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.72, new_layer_p, 1, -0.30000000000000004, 0.77, new_layer_p, 1, -0.28, 0.87, new_layer_p, 1, -0.26, 0.9400000000000001, new_layer_p, 1, -0.24, 1.02, new_layer_p, 1, -0.2, 1.13, new_layer_p, 1, -0.12, 1.32]",-10.2962,-10.63
0,er_graph_2,"[[1, 2], 0.09, [1, 3], 0.26, [1, 4], 0.13, [1, 5], 0.29, [1, 6], 0.71, [1, 9], 0.51, [1, 10], 0.15, [2, 3], 0.28, [2, 4], 0.16, [2, 5], 0.31, [2, 6], 0.17, [2, 7], 0.55, [2, 8], 0.17, [2, 9], 0.89, [2, 10], 0.28, [3, 4], 0.72, [3, 5], 0.39, [3, 6], 0.28, [3, 8], 0.65, [3, 9], 0.56, [3, 10], 0.55, [4, 5], 0.46, [4, 7], 0.92, [4, 8], 0.39, [4, 9], 0.4, [4, 10], 0.07, [5, 6], 0.84, [5, 7], 0.48, [5, 8], 0.59, [5, 9], 0.64, [5, 10], 0.12, [6, 7], 0.24, [6, 8], 0.22, [6, 9], 0.07, [7, 8], 0.12, [7, 10], 0.77, [8, 9], 0.13, [9, 10], 0.26]",38,"[new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.71, new_layer_p, 1, -0.30000000000000004, 0.77, new_layer_p, 1, -0.28, 0.89, new_layer_p, 1, -0.26, 0.96, new_layer_p, 1, -0.23, 1.05, new_layer_p, 1, -0.18, 1.15, new_layer_p, 1, -0.1, 1.35]",-10.329601,-10.63
0,er_graph_2,"[[1, 2], 0.09, [1, 3], 0.26, [1, 4], 0.13, [1, 5], 0.29, [1, 6], 0.71, [1, 9], 0.51, [1, 10], 0.15, [2, 3], 0.28, [2, 4], 0.16, [2, 5], 0.31, [2, 6], 0.17, [2, 7], 0.55, [2, 8], 0.17, [2, 9], 0.89, [2, 10], 0.28, [3, 4], 0.72, [3, 5], 0.39, [3, 6], 0.28, [3, 8], 0.65, [3, 9], 0.56, [3, 10], 0.55, [4, 5], 0.46, [4, 7], 0.92, [4, 8], 0.39, [4, 9], 0.4, [4, 10], 0.07, [5, 6], 0.84, [5, 7], 0.48, [5, 8], 0.59, [5, 9], 0.64, [5, 10], 0.12, [6, 7], 0.24, [6, 8], 0.22, [6, 9], 0.07, [7, 8], 0.12, [7, 10], 0.77, [8, 9], 0.13, [9, 10], 0.26]",38,"[new_layer_p, 1, -0.48, 0.35000000000000003, new_layer_p, 1, -0.35000000000000003, 0.71, new_layer_p, 1, -0.30000000000000004, 0.77, new_layer_p, 1, -0.28, 0.87, new_layer_p, 1, -0.26, 0.92, new_layer_p, 1, -0.24, 0.98, new_layer_p, 1, -0.22, 1.06, new_layer_p, 1, -0.18, 1.2, new_layer_p, 1, -0.1, 1.43]",-10.32131,-10.63
0,er_graph_2,"[[1, 2], 0.09, [1, 3], 0.26, [1, 4], 0.13, [1, 5], 0.29, [1, 6], 0.71, [1, 9], 0.51, [1, 10], 0.15, [2, 3], 0.28, [2, 4], 0.16, [2, 5], 0.31, [2, 6], 0.17, [2, 7], 0.55, [2, 8], 0.17, [2, 9], 0.89, [2, 10], 0.28, [3, 4], 0.72, [3, 5], 0.39, [3, 6], 0.28, [3, 8], 0.65, [3, 9], 0.56, [3, 10], 0.55, [4, 5], 0.46, [4, 7], 0.92, [4, 8], 0.39, [4, 9], 0.4, [4, 10], 0.07, [5, 6], 0.84, [5, 7], 0.48, [5, 8], 0.59, [5, 9], 0.64, [5, 10], 0.12, [6, 7], 0.24, [6, 8], 0

This computes how close each QAOA-GPT circuit’s energy is to the optimal solution (AR — approximation ratio).

If the ratio = 1 → perfect solution.

If <1 → circuit energy is above the optimal energy (since MaxCut is a minimization problem in some conventions, sometimes they flip it; conceptually, ratio shows performance).

In [24]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(0.966918413140715)

In [25]:
# on avg, how many layers are there in the predicted circuits
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(8.448)